# flydnet Colab 테스트

1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** (CPU만으로도 동작은 함)
2. 위에서부터 차례로 실행 (Shift+Enter)

In [ ]:
import torch, platform
print("python", platform.python_version(), "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로 실행)")

In [ ]:
!pip install -q flydnet

In [ ]:
!python -m flydnet download
!python -m flydnet

## 버섯체: 순전파

In [ ]:
import time, torch, flydnet as fd
print("flydnet", fd.__version__)
mb = fd.Circuit.from_flywire()
print(mb)
layer = fd.torch.ConnectomeLayer(mb, "PN", "KC", t_ms=50, gains={"PN>KC": 2.0}, input_mode="regular")
t = time.time()
kc = layer(torch.rand(64, 344) * 100)
if torch.cuda.is_available(): torch.cuda.synchronize()
print("KC", tuple(kc.shape), f"켜진 비율 {(kc > 0).float().mean():.1%}", f"{time.time() - t:.2f}s")

## 버섯체: 역전파 학습 (연결 세기)

In [ ]:
import torch.nn as nn
layer = fd.torch.ConnectomeLayer(mb, "PN", "KC", t_ms=50, dt=0.5, gains={"PN>KC": 2.0},
                           input_mode="regular", trainable=True)
model = nn.Sequential(fd.torch.RateEncoder(784, 344), layer,
                      nn.BatchNorm1d(layer.n_out), nn.Linear(layer.n_out, 10)).to(layer.dev)
opt = torch.optim.Adam([{"params": [layer.log_scale], "lr": 3e-2},
                        {"params": model[-1].parameters(), "lr": 3e-3}])
x, y = torch.rand(64, 784, device=layer.dev), torch.randint(0, 10, (64,), device=layer.dev)
for step in range(5):
    t = time.time()
    loss = nn.functional.cross_entropy(model(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
    print(f"step {step}: loss {loss.item():.3f}, {time.time() - t:.2f}s")
print("연결 세기 기울기 정상:", torch.isfinite(layer.log_scale.grad).all().item())

## 시각계: 운동 감지 경로, 연속값 뉴런

In [ ]:
import numpy as np
vc_full = fd.visual_circuit()
xy_full = fd.torch.column_map(vc_full)
keep = np.concatenate([vc_full.groups[g] for g in fd.MOTION_PATHWAY if g in vc_full.groups])
vc, xy = vc_full.subset(fd.MOTION_PATHWAY), xy_full[keep]
print(vc)
print("배선 속 방향 구조 (T4, 도/일관성):", {k: (round(a), round(r, 2)) for k, (a, r) in fd.direction_offsets(vc, xy).items()})

OUT = ["T4a", "T4b", "T4c", "T4d"]
L = fd.torch.ConnectomeLayer(vc.normalized(), fd.PHOTORECEPTORS, OUT, t_ms=300, dt=2.0, neuron="graded",
                       params={"w_syn": 3.0}, bias={g: (0.0 if g in fd.PHOTORECEPTORS else 0.2) for g in vc.groups},
                       count_from_ms=50, trainable=True, share="pair", train_neurons=True)
lum = fd.torch.drifting_grating(xy[L.in_idx.cpu().numpy()], [0, 90, 180, 270], 300, 150, wavelength=6, temporal_hz=4)
t = time.time()
out = L(0.5 * (1 + lum))
out.sum().backward()
print("T4 출력", tuple(out.shape), f"순전파+역전파 {time.time() - t:.2f}s")

## 무작위 배선 대조군

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sh = vc.shuffled(seed=0)
    merge = {f"T{k}{d}": f"T{k}" for k in "45" for d in "abcd"}
    loc = vc.shuffled(seed=0, local=(xy, 2.0), merge=merge)
for name, c in [("실제", vc), ("국소 무작위", loc), ("전체 무작위", sh)]:
    r = np.mean([v[1] for v in fd.direction_offsets(c, xy).values()])
    print(f"{name}: T4 방향 일관성 {r:.2f}")
print("모두 통과하면 flydnet이 이 환경에서 정상 동작함")